# 🏥 SafeRx AI — Fase 1: Generación y Extracción de Datos Clínicos

**Proyecto:** SafeRx AI — Asistente Clínico Inteligente  
**Cliente:** FritzeFriends (Red de 3 hospitales regionales y 12 clínicas)  
**Módulo:** Ingesta, Integridad Referencial y Extracción de Datos Crudos  
**Autor:** Equipo de Ciencia de Datos & IA Clínica SafeRx  

---

## 📋 Contexto y Objetivos de la Fase

El grupo hospitalario **FritzeFriends** atiende aproximadamente 3,000 consultas diarias. En pacientes geriátricos y con comorbilidades (polifarmacia), el cruce de 4 a 7 medicamentos en los 15 minutos que dura una consulta médica genera un alto riesgo de interacciones farmacológicas no detectadas.

En este notebook:
1. Conectamos con el motor relacional SQLite (`database/hospital.db`).
2. Validamos el esquema relacional y la integridad de las entidades (`Pacientes`, `Medicos`, `Medicamentos`, `Consultas`, `Recetas`, `Interacciones`).
3. Evaluamos la coherencia clínica de las distribuciones (edad, género, diagnósticos y principios activos).
4. Exportamos los conjuntos de datos en crudo (*raw data*) a `data/raw/` para garantizar la reproducibilidad de los análisis posteriores.


In [ ]:
import os
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np

# Configuración de rutas
BASE_DIR = Path("..").resolve()
DB_PATH = BASE_DIR / "database" / "hospital.db"
RAW_DATA_DIR = BASE_DIR / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Base de datos objetivo: {DB_PATH}")
print(f"✅ Directorio raw: {RAW_DATA_DIR}")


## 1. Conexión y Extracción de Tablas Relacionales

Extraemos cada tabla del modelo relacional en un DataFrame independiente para auditar su estructura.

In [ ]:
# Conexión a SQLite
conn = sqlite3.connect(DB_PATH)

# Lectura de tablas maestras
df_pacientes = pd.read_sql("SELECT * FROM Pacientes", conn)
df_medicos = pd.read_sql("SELECT * FROM Medicos", conn)
df_medicamentos = pd.read_sql("SELECT * FROM Medicamentos", conn)
df_consultas = pd.read_sql("SELECT * FROM Consultas", conn)
df_recetas = pd.read_sql("SELECT * FROM Recetas", conn)
df_interacciones = pd.read_sql("SELECT * FROM Interacciones", conn)

conn.close()

tablas_info = pd.DataFrame({
    "Tabla": ["Pacientes", "Medicos", "Medicamentos", "Consultas", "Recetas", "Interacciones"],
    "Registros": [len(df_pacientes), len(df_medicos), len(df_medicamentos), 
                  len(df_consultas), len(df_recetas), len(df_interacciones)],
    "Columnas": [len(df_pacientes.columns), len(df_medicos.columns), len(df_medicamentos.columns),
                 len(df_consultas.columns), len(df_recetas.columns), len(df_interacciones.columns)]
})

display(tablas_info)


## 2. Inspección Clínica de Entidades

### 2.1. Cohorte de Pacientes
Revisamos la distribución etaria y la asignación de patologías previas de acuerdo a la lógica clínica.

In [ ]:
print("--- Muestra de Pacientes ---")
display(df_pacientes.head(5))

print("\n--- Resumen Demográfico de Pacientes ---")
display(df_pacientes[["edad"]].describe().T)

print("\n--- Distribución de Condiciones Previas por Género ---")
display(pd.crosstab(df_pacientes["condiciones_previas"], df_pacientes["genero"], margins=True))


### 2.2. Médicos y Especialidades
El personal médico del grupo hospitalario está distribuido en áreas clave donde la prescripción es más crítica: Urgencias, Medicina General y Geriatría.

In [ ]:
print("--- Distribución de Especialidades Médicas ---")
display(df_medicos["especialidad"].value_counts().to_frame(name="Total Médicos"))


### 2.3. Vademécum y Matriz de Interacciones Peligrosas
Analizamos los principios activos catalogados y las interacciones de alto riesgo que la IA debe vigilar.

In [ ]:
print("--- Catálogo de Medicamentos ---")
display(df_medicamentos)

# Cruce descriptivo de interacciones
df_interac_detallado = df_interacciones.merge(
    df_medicamentos[["id_medicamento", "principio_activo"]], 
    left_on="id_medicamento_1", right_on="id_medicamento"
).rename(columns={"principio_activo": "medicamento_A"}).drop(columns=["id_medicamento"])

df_interac_detallado = df_interac_detallado.merge(
    df_medicamentos[["id_medicamento", "principio_activo"]], 
    left_on="id_medicamento_2", right_on="id_medicamento"
).rename(columns={"principio_activo": "medicamento_B"}).drop(columns=["id_medicamento"])

print("\n--- Matriz de Interacciones Clínicas Conocidas ---")
display(df_interac_detallado[["medicamento_A", "medicamento_B", "gravedad", "descripcion"]])


## 3. Integridad Referencial y Coherencia de Recetas

Verificamos que no existan recetas huérfanas ni medicamentos inexistentes en las prescripciones.

In [ ]:
# Verificación de integridad
recetas_sin_consulta = ~df_recetas["id_consulta"].isin(df_consultas["id_consulta"])
recetas_sin_medicamento = ~df_recetas["id_medicamento"].isin(df_medicamentos["id_medicamento"])

print(f"Recetas con id_consulta inválido: {recetas_sin_consulta.sum()}")
print(f"Recetas con id_medicamento inválido: {recetas_sin_medicamento.sum()}")

assert recetas_sin_consulta.sum() == 0, "Error de integridad en recetas -> consultas"
assert recetas_sin_medicamento.sum() == 0, "Error de integridad en recetas -> medicamentos"
print("✅ Integridad referencial 100% verificada.")


## 4. Exportación de Datos Crudos (`data/raw/`)

Exportamos los archivos CSV crudos para versionar las fuentes de datos del proyecto.

In [ ]:
df_pacientes.to_csv(RAW_DATA_DIR / "pacientes.csv", index=False)
df_medicos.to_csv(RAW_DATA_DIR / "medicos.csv", index=False)
df_medicamentos.to_csv(RAW_DATA_DIR / "medicamentos.csv", index=False)
df_consultas.to_csv(RAW_DATA_DIR / "consultas.csv", index=False)
df_recetas.to_csv(RAW_DATA_DIR / "recetas.csv", index=False)
df_interacciones.to_csv(RAW_DATA_DIR / "interacciones.csv", index=False)

print("Archivos exportados exitosamente en:")
for f in RAW_DATA_DIR.glob("*.csv"):
    print(f" - {f.name} ({f.stat().st_size / 1024:.1f} KB)")


## 5. Conclusiones y Próximos Pasos
 
1. **Datos listos y consistentes:** Se confirmaron 300 pacientes con fenotipos clínicos reales, 12 médicos especialistas, 94 principios activos reales, 57 interacciones documentadas, más de 900 consultas y 525 reportes reales de OpenFDA (FAERS).
2. **Siguiente fase (`02_exploratory_analysis.ipynb`):** Procederemos a realizar el Análisis Exploratorio de Datos (EDA) para cuantificar la prevalencia de polifarmacia, identificar perfiles de riesgo y evaluar la frecuencia de incidentes farmacológicos.
